<a href="https://colab.research.google.com/github/orshraga/DIV2K_SR_challenge/blob/signal-processing-voises/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# Deep Learning and its Applications to Signal and Image Processing
# FINAL PROJECT - DIV2K Super Resolution Challenge
#
# Course: 361.2.1120
# Authors:   Or Shraga , Gal Apple
# # ================================================================

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Corrected path: include the folder within My Drive
project_path = '/content/drive/My Drive/DIV2K_SR_chalge'
os.chdir(project_path)
print("✅ Current working directory:", os.getcwd())

# List files to confirm
!ls -la

# Install requirements if the file exists
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt
else:
    print("⚠️ requirements.txt not found in this directory. Skipping install.")


!pip install --upgrade kaggle
!pip install pytorch-lightning
!pip install -r requirements.txt

In [ ]:
import torch
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import os
from plot import (
    visualize_ychannel_samples,
    show_sample_images,
    plot_training_loss,
    show_sr_examples,
    plot_multiple_training_losses
)
# from model import ESPCNLightning_YChannel,ESPCNLightning_YChannel_RES
from model import ESPCNLightning_YChannel

from utils import set_seed, seed_worker
from eval import evaluate_model_ycbcr, compare_models_examples
from data import (
    use_seed_and_split,
    DIV2K_YCbCr_SR_Dataset,
    get_image_files,
    download_kaggle_dataset,
)
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from train_utils import run_model, run_model_multiple_seeds


In [ ]:
current_dir = os.getcwd()
KAGGLE_JSON = os.path.join(current_dir, "kaggle.json")
DATASET = "joe1995/div2k-dataset"
DOWNLOAD_DIR = "./div2k"

RUN_THREE_SEEDS = False
RUN_ONE_SEED = True
RUN_ABLATION= False
# SEEDS = [42, 123, 999]
SEEDS = [42, 123]

# used for all models
SEED = 42
LR = 1e-4
SCALE_FACTOR = 4
VAL_BATCH_SIZE = 4
BATCH_SIZE = 16

EPOCHS_A = 100
split_ratio_A=0.9

# used for residual and "no_global" abletion model
split_ratio_B=0.9
EPOCHS_B = 100

In [ ]:
# Remove old partial zip if exists
zip_path = './div2k/div2k-dataset.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
    print("⚠️ Existing partial zip file deleted.")

# Download dataset
download_kaggle_dataset(
    dataset=DATASET,
    kaggle_json_path=KAGGLE_JSON,
    download_dir=DOWNLOAD_DIR
)

# Get files
files = get_image_files(DOWNLOAD_DIR, subset="train")
test_files = get_image_files(DOWNLOAD_DIR, subset="valid")

show_sample_images(files, num=4, title_prefix="Train")
show_sample_images(test_files, num=2, title_prefix="Val")

# 🚀 Run Three Different Seeds and Print Report
Calculate mean and standard deviation of main evaluation metric


In [ ]:
if RUN_THREE_SEEDS:
  results_base = run_model_multiple_seeds(
      files, test_files,
      seeds=SEEDS,
      split_ratio=split_ratio_A,
      max_epochs=EPOCHS_A,
      scale=SCALE_FACTOR,
      lr=LR,
      batch_size_train=BATCH_SIZE,
      batch_size_val=VAL_BATCH_SIZE,
      model_type='base',
  )

  results_residual = run_model_multiple_seeds(
      files, test_files,
      seeds=SEEDS,
      split_ratio=split_ratio_B,
      max_epochs=EPOCHS_B,
      scale=SCALE_FACTOR,
      lr=LR,
      batch_size_train=BATCH_SIZE,
      batch_size_val=VAL_BATCH_SIZE,
      model_type='residual',
  )


# 🚀 Run Seed and Print Report Model 1 & Model 1
Calculate mean and standard deviation of main evaluation metric


In [ ]:
if RUN_ONE_SEED:
    all_train_losses, all_val_losses = [], []

    # Run base (vanilla) model
    results_vanilla = run_model(
        files, test_files,
        seed=SEED,
        split_ratio=split_ratio_A,
        max_epochs=EPOCHS_A,
        scale=SCALE_FACTOR,
        lr=LR,
        batch_size_train=BATCH_SIZE,
        batch_size_val=VAL_BATCH_SIZE,
        model_type='base',
    )
    vanilla_model = results_vanilla['model']
    all_train_losses.append(results_vanilla['train_losses'])
    all_val_losses.append(results_vanilla['val_losses'])

    # Run residual model
    results_residual = run_model(
        files, test_files,
        seed=SEED,
        split_ratio=split_ratio_A,
        max_epochs=EPOCHS_B,
        scale=SCALE_FACTOR,
        lr=LR,
        batch_size_train=BATCH_SIZE,
        batch_size_val=VAL_BATCH_SIZE,
        model_type='residual',
    )
    residual_model = results_residual['model']
    all_train_losses.append(results_residual['train_losses'])
    all_val_losses.append(results_residual['val_losses'])

    # Plot both losses: vanilla vs. residual
    plot_multiple_training_losses(
        all_train_losses,
        all_val_losses
    )

    # Print comparison table
    print("\n=== Comparison Table ===")
    print(f"{'Metric':<10} {'Residual':<12} {'Vanilla':<12}")
    for metric in ['psnr', 'ssim', 'fid']:
        print(f"{metric.upper():<10} {results_residual[metric]:<12.4f} {results_vanilla[metric]:<12.4f}")

    # Create test dataset (once!)
    test_ds = DIV2K_YCbCr_SR_Dataset(
        test_files, scale=SCALE_FACTOR, mode='test', add_cbcr=True
    )

    # Visual comparison: residual vs. vanilla
    compare_models_examples(
        model1=vanilla_model,
        model2=residual_model,
        dataset=test_ds,
        device='cuda',
        threshold_good=22,
        max_images=200
    )


# 🚀 Run Seed and Print Report Model 2 (Residual) & Model 2 (Withut global connection-Ablation study)
Calculate mean and standard deviation of main evaluation metric


In [ ]:
if RUN_ABLATION:
    all_train_losses, all_val_losses = [], []

    # Run residual model
    results_residual = run_model(
        files, test_files,
        seed=SEED,
        split_ratio=split_ratio_A,
        max_epochs=EPOCHS_B,
        scale=SCALE_FACTOR,
        lr=LR,
        batch_size_train=BATCH_SIZE,
        batch_size_val=VAL_BATCH_SIZE,
        model_type='residual',
    )
    residual_model = results_residual['model']
    all_train_losses.append(results_residual['train_losses'])
    all_val_losses.append(results_residual['val_losses'])

    # Run ablation model (no_global)
    results_ablation = run_model(
        files, test_files,
        seed=SEED,
        split_ratio=split_ratio_A,
        max_epochs=EPOCHS_B,
        scale=SCALE_FACTOR,
        lr=LR,
        batch_size_train=BATCH_SIZE,
        batch_size_val=VAL_BATCH_SIZE,
        model_type='no_global',
    )
    ablation_model = results_ablation['model']
    all_train_losses.append(results_ablation['train_losses'])
    all_val_losses.append(results_ablation['val_losses'])

    # Plot both losses: residual vs. ablation
    plot_multiple_training_losses(
        all_train_losses,
        all_val_losses
    )

    # Print comparison table
    print("\n=== Comparison Table ===")
    print(f"{'Metric':<10} {'Residual':<12} {'Ablation':<12}")
    for metric in ['psnr', 'ssim', 'fid']:
        print(f"{metric.upper():<10} {results_residual[metric]:<12.4f} {results_ablation[metric]:<12.4f}")

    # Create test dataset
    test_ds = DIV2K_YCbCr_SR_Dataset(
        test_files, scale=SCALE_FACTOR, mode='test', add_cbcr=True
    )

    # Visual comparison: residual vs. ablation
    compare_models_examples(
        model1=residual_model,
        model2=ablation_model,
        dataset=test_ds,
        device='cuda',
        threshold_good=22,  # set to realistic PSNR threshold
        max_images=200
    )
